# LSTM vs GRU — AAPL Stock Price Prediction
Same pipeline as the RNN notebook, but with **LSTM** and **GRU** models trained in parallel, then compared side-by-side on the test set.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 1. Load & Preprocess Data

In [2]:
# Load data
data = pd.read_csv('top50_adjclose_2010_2025.csv')
apple_data = data[['Date', 'AAPL']].copy()
apple_data['Date'] = pd.to_datetime(apple_data['Date'])
apple_data = apple_data.dropna().reset_index(drop=True)

apple_prices = apple_data['AAPL'].values.reshape(-1, 1)

def create_sequences(data, lookback=60):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(data[i])
    return np.array(X), np.array(y)

lookback = 60
X_raw, y_raw = create_sequences(apple_prices, lookback)

train_size = int(0.7  * len(X_raw))
val_size   = int(0.15 * len(X_raw))

X_train_raw, y_train_raw = X_raw[:train_size],                    y_raw[:train_size]
X_val_raw,   y_val_raw   = X_raw[train_size:train_size+val_size], y_raw[train_size:train_size+val_size]
X_test_raw,  y_test_raw  = X_raw[train_size+val_size:],           y_raw[train_size+val_size:]

scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train_raw.reshape(-1, 1)).reshape(X_train_raw.shape)
y_train = scaler.transform(y_train_raw)

X_val = scaler.transform(X_val_raw.reshape(-1, 1)).reshape(X_val_raw.shape)
y_val = scaler.transform(y_val_raw)

X_test = scaler.transform(X_test_raw.reshape(-1, 1)).reshape(X_test_raw.shape)
y_test = scaler.transform(y_test_raw)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')
print(f'Input shape: {X_train.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'top50_adjclose_2010_2025.csv'

## 2. Dataset & DataLoaders

In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(StockDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(StockDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

## 3. Model Definitions — LSTM & GRU

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # last timestep


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])   # last timestep


lstm_model = LSTMModel().to(device)
gru_model  = GRUModel().to(device)

print('LSTM parameters:', sum(p.numel() for p in lstm_model.parameters()))
print('GRU  parameters:', sum(p.numel() for p in gru_model.parameters()))

## 4. Shared Training Function

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, model_name='Model'):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
        train_loss = epoch_loss / len(train_loader.dataset)

        # --- Validate ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                val_loss += criterion(pred, y_batch).item() * len(X_batch)
        val_loss /= len(val_loader.dataset)

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(f'[{model_name}] Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}')

    # Restore best weights
    model.load_state_dict(best_state)
    print(f'[{model_name}] Best val loss: {best_val_loss:.6f}\n')
    return train_losses, val_losses

## 5. Train LSTM

In [ ]:
EPOCHS = 50

lstm_train_losses, lstm_val_losses = train_model(
    lstm_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, model_name='LSTM'
)

## 6. Train GRU

In [ ]:
gru_train_losses, gru_val_losses = train_model(
    gru_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, model_name='GRU'
)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, train_l, val_l, name, color in zip(
    axes,
    [lstm_train_losses, gru_train_losses],
    [lstm_val_losses,   gru_val_losses],
    ['LSTM', 'GRU'],
    ['#0071e3', '#ff6b35']
):
    ax.plot(train_l, label='Train', color=color, linewidth=2)
    ax.plot(val_l,   label='Val',   color=color, linewidth=2, linestyle='--')
    ax.set_title(f'{name} — Loss Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds_s, actuals_s = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds_s.append(model(X_batch).cpu().numpy())
            actuals_s.append(y_batch.numpy())
    preds_scaled   = np.concatenate(preds_s,   axis=0)
    actuals_scaled = np.concatenate(actuals_s, axis=0)
    preds   = scaler.inverse_transform(preds_scaled)
    actuals = scaler.inverse_transform(actuals_scaled)
    return preds, actuals

lstm_preds, actuals = evaluate(lstm_model, test_loader)
gru_preds,  _       = evaluate(gru_model,  test_loader)

# Test dates
test_start_idx = train_size + val_size + lookback
test_dates = apple_data['Date'].iloc[test_start_idx:test_start_idx + len(actuals)].values

def metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'{name:6s} | MAE: ${mae:.2f} | RMSE: ${rmse:.2f} | MAPE: {mape:.2f}%')
    return mae, rmse, mape

print('Model  | MAE         | RMSE        | MAPE')
print('-' * 50)
lstm_metrics = metrics('LSTM', actuals, lstm_preds)
gru_metrics  = metrics('GRU',  actuals, gru_preds)

## 9. Predicted vs Actual — LSTM & GRU Comparison

In [ ]:
plt.figure(figsize=(18, 7))

plt.plot(test_dates, actuals,    label='Actual',    color='black',   linewidth=2.5)
plt.plot(test_dates, lstm_preds, label='LSTM Pred', color='#0071e3', linewidth=2, linestyle='--')
plt.plot(test_dates, gru_preds,  label='GRU Pred',  color='#ff6b35', linewidth=2, linestyle='-.')

plt.title('LSTM vs GRU — Test Set Predictions vs Actual (AAPL)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=13)
plt.ylabel('AAPL Price ($)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Annotate metrics in the plot
textstr = (
    f"LSTM | MAE: ${lstm_metrics[0]:.2f} | RMSE: ${lstm_metrics[1]:.2f} | MAPE: {lstm_metrics[2]:.2f}%\n"
    f"GRU  | MAE: ${gru_metrics[0]:.2f}  | RMSE: ${gru_metrics[1]:.2f}  | MAPE: {gru_metrics[2]:.2f}%"
)
props = dict(boxstyle='round', facecolor='white', alpha=0.7)
plt.gca().text(
    0.01, 0.97, textstr,
    transform=plt.gca().transAxes,
    fontsize=10, verticalalignment='top', bbox=props
)

plt.tight_layout()
plt.show()

## 10. Individual Subplots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 12), sharex=True)

for ax, preds, name, color in zip(
    axes,
    [lstm_preds, gru_preds],
    ['LSTM', 'GRU'],
    ['#0071e3', '#ff6b35']
):
    ax.plot(test_dates, actuals, label='Actual', color='black', linewidth=2)
    ax.plot(test_dates, preds, label=f'{name} Pred', color=color, linewidth=2, linestyle='--')
    ax.set_title(f'{name} — Test Set Predictions vs Actual', fontsize=14, fontweight='bold')
    ax.set_ylabel('AAPL Price ($)', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=12)
plt.tight_layout()
plt.show()